In [43]:
# Importing the essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Load the dataset
file_path = '/workspace/oscar/the_oscar_award.csv'  # Adjust the path as necessary
df = pd.read_csv(file_path)

# Check the first few rows of the dataset
df.head()


In [ ]:
# Get basic information about the dataset
df.info()

# Get descriptive statistics (for numerical columns)
df.describe()

# Check for missing values
df.isnull().sum()


In [ ]:
# Check the column names
df.columns

# Look at the unique values in each column (useful for categorical columns)
for column in df.select_dtypes(include=['object']).columns:
    print(f"{column}: {df[column].nunique()} unique values")
    print(df[column].unique())
    print("\n")


In [ ]:
# Filter the data for a specific year (e.g., 1928)
subset_data = df[df['year_ceremony'] >= 2000]

# Display all the filtered data
import pandas as pd
pd.set_option('display.max_rows', None)  # This ensures that all rows are displayed

display(subset_data) # Display the filtered data

In [49]:
# Filter the data for a specific year (e.g., 1928)
subset_data = df[df['year_ceremony'] >= 2000]

# Output the filtered data to a new CSV file
output_file_path = '/workspace/oscar/base-year-2000.csv'  # Replace with your desired output path
subset_data.to_csv(output_file_path, index=False)  # index=False prevents writing row numbers


In [ ]:
# Importing the essential libraries
import pandas as pd

# Load the dataset
file_path = '/workspace/oscar/the_oscar_award.csv'  # Adjust the path as necessary
df = pd.read_csv(file_path)

# Step 1: Filter the data for year 2000 or after
subset_data = df[df['year_ceremony'] >= 2000]

# Step 2: Filter to only keep rows where the category is ACTOR IN LEADING ROLE, ACTRESS IN LEADING ROLE, or BEST PICTURE
categories_of_interest = ['ACTOR IN A LEADING ROLE', 'ACTRESS IN A LEADING ROLE', 'BEST PICTURE']

subset_data_filtered = subset_data[subset_data['category'].isin(categories_of_interest)]

# Output the filtered data to a new CSV file
output_file_path = '/workspace/oscar/base-year-2000-filtered.csv'  # Replace with your desired output path
subset_data_filtered.to_csv(output_file_path, index=False)  # index=False prevents writing row numbers

# Display the filtered data
import pandas as pd
pd.set_option('display.max_rows', None)  # This ensures that all rows are displayed
display(subset_data_filtered)  # Display the filtered data


In [ ]:
# Load the newly saved CSV into a DataFrame
output_file_path = '/workspace/oscar/base-year-2000-filtered.csv'  # Replace with the path to your output CSV
year2000_df = pd.read_csv(output_file_path)

# Display the first few rows of the new DataFrame
display(year2000_df)  # Or you can use print(new_df) if you prefer


In [ ]:
!pip install IMDbPY

In [ ]:
pip install requests

In [ ]:
import pandas as pd
from imdb import Cinemagoer
import time
import requests

# Create Cinemagoer object
ia = Cinemagoer()

# Define OMDb API key (you can get a free API key at https://www.omdbapi.com/)
OMDB_API_KEY = '2121a3ae'

# Read CSV file
year2000_df = pd.read_csv('/workspace/oscar/base-year-2000-filtered.csv')  # Replace with your file path

# Create new columns for IMDb ratings and box office
year2000_df['imdb_rating'] = None
year2000_df['box_office'] = None

# Set to track movies we've already processed
processed_movies = set()

# Function to fetch box office data from OMDb API
def get_box_office(movie_name):
    try:
        # Request movie data from OMDb API
        response = requests.get(f"http://www.omdbapi.com/?t={movie_name}&apikey={OMDB_API_KEY}")
        data = response.json()
        
        if data['Response'] == 'True':
            return data.get('BoxOffice', None)  # Return BoxOffice value if available
        else:
            print(f"Error fetching box office for {movie_name}: {data.get('Error', 'Unknown error')}")
            return None
    except Exception as e:
        print(f"Error fetching box office for {movie_name}: {e}")
        return None

# Iterate through each film in year2000_df DataFrame
for index, row in year2000_df.iterrows():
    movie_name = row['film']
    
    # Check if movie_name is valid (not NaN or empty)
    if not isinstance(movie_name, str) or not movie_name.strip():
        print(f"Skipping invalid movie name at index {index}: {movie_name}")
        continue  # Skip empty or invalid movie names
    
    # Skip duplicate movies (those already processed)
    if movie_name in processed_movies:
        print(f"Skipping already processed movie: {movie_name}")
        continue  # Skip already processed movies
    
    try:
        # Search for the movie
        movies = ia.search_movie(movie_name)
        
        if not movies:
            print(f"No results found for: {movie_name}")
            continue
            
        # Get first result and fetch movie details
        movie = ia.get_movie(movies[0].movieID)
        
        # Extract IMDb rating if available
        if 'rating' in movie.keys():
            year2000_df.at[index, 'imdb_rating'] = movie['rating']
            print(f"Found rating {movie['rating']} for {movie_name}")
        else:
            print(f"No rating available for: {movie_name}")
        
        # Fetch box office data from OMDb API
        box_office = get_box_office(movie_name)
        if box_office:
            year2000_df.at[index, 'box_office'] = box_office
            print(f"Found box office {box_office} for {movie_name}")
        else:
            print(f"No box office data available for: {movie_name}")
        
        # Mark this movie as processed
        processed_movies.add(movie_name)
            
    except Exception as e:
        print(f"Error processing {movie_name}: {str(e)}")
    
    # Add delay to prevent rate limiting
    time.sleep(1)  # Be nice to IMDb's servers

# Save updated DataFrame to a new CSV
year2000_df.to_csv('/workspace/oscar/updated_movies_with_ratings_and_box_office.csv', index=False)

# Display the first few rows of the updated DataFrame
year2000_df.head()  # You can also use display() in Jupyter if needed


In [75]:
import pandas as pd

# Load the CSV file into a DataFrame (update the filename as needed)
df = pd.read_csv('/workspace/oscar/updated_movies_with_ratings_and_box_office.csv')

# Replace empty strings with NaN (if missing values are stored as empty strings)
df['imdb_rating'] = df['imdb_rating'].replace('', pd.NA)
df['box_office'] = df['box_office'].replace('', pd.NA)

# Define a helper function to fill missing values within each group
def fill_missing(series):
    non_missing = series.dropna()
    if not non_missing.empty:
        # Fill all missing values in the series with the first non-missing value
        return series.fillna(non_missing.iloc[0])
    return series

# Assuming the film name column is named 'film'; change this if necessary.
df['imdb_rating'] = df.groupby('film')['imdb_rating'].transform(fill_missing)
df['box_office'] = df.groupby('film')['box_office'].transform(fill_missing)

# Optionally, save the updated DataFrame to a new CSV file
df.to_csv('updated_movies_with_ratings_and_box_office1.csv', index=False)

print("Missing values for 'imdb_ranting' and 'box_office' have been filled based on duplicate film entries.")


Missing values for 'imdb_ranting' and 'box_office' have been filled based on duplicate film entries.


In [ ]:
import pandas as pd
import requests

API_KEY = "2121a3ae"  # Replace with your actual API key

def get_metascore_from_omdb(title):
    url = f"http://www.omdbapi.com/?t={title}&apikey={API_KEY}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return data.get("Metascore", "N/A")
    except Exception as e:
        print(f"Error fetching data for {title}: {e}")
    return "N/A"

# Load your CSV (ensure it has a 'film' column)
df = pd.read_csv("/workspace/oscar/updated_movies_with_ratings_and_box_office1.csv")
df["Metascore"] = df["film"].apply(get_metascore_from_omdb)
df.to_csv("movies_with_metascore.csv", index=False)

In [78]:
import pandas as pd
import requests

API_KEY = "2121a3ae"  # Replace with your actual API key

def get_rotten_tomatoes_rating(title):
    url = f"http://www.omdbapi.com/?t={title}&apikey={API_KEY}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            ratings = data.get("Ratings", [])
            for rating in ratings:
                if rating.get("Source") == "Rotten Tomatoes":
                    return rating.get("Value")
    except Exception as e:
        print(f"Error fetching data for {title}: {e}")
    return "N/A"

# Load your CSV (ensure it has a 'film' column)
df = pd.read_csv("/workspace/oscar/movies_with_metascore.csv")

# Apply the function to get Rotten Tomatoes rating
df["Tomatometer"] = df["film"].apply(get_rotten_tomatoes_rating)

# Save the updated dataframe to a new CSV file
df.to_csv("movies_with_metascore_and_tomatometer.csv", index=False)


In [80]:
import pandas as pd

# Create a DataFrame from the data
df = pd.read_csv("/workspace/oscar/movies_with_metascore_and_tomatometer.csv")

# Filter rows where category is "BEST PICTURE"
df_best_picture = df[df["category"] == "BEST PICTURE"]

# Check if columns exist before dropping
columns_to_drop = ["year_film", "year_ceremony", "box_office", "name"]
columns_existing = [col for col in columns_to_drop if col in df.columns]

# Drop the columns that exist
df_best_picture_cleaned = df_best_picture.drop(columns=columns_existing)

# Save the cleaned DataFrame to a CSV file
output_file_path = 'cleaned_best_picture_2000.csv'
df_best_picture_cleaned.to_csv(output_file_path, index=False)

# Display the path to the saved CSV
print(f"The cleaned CSV has been saved to: {output_file_path}")


The cleaned CSV has been saved to: cleaned_best_picture_2000.csv


In [100]:
import pandas as pd

# Load the cleaned data
df = pd.read_csv("/workspace/oscar/cleaned_best_picture_2000.csv")

# Check for missing values in each column
missing_values = df.isnull().sum()

# Display the missing values count for each column
print("Missing values count for each column:")
print(missing_values)


# Check the data types of each column
print("\nData types of each column:")
print(df.dtypes)

# Get summary statistics for numerical columns
print("\nSummary statistics for numerical columns:")
print(df.describe())

# Get counts for categorical columns
print("\nCounts for categorical columns:")
print(df['category'].value_counts())


Missing values count for each column:
ceremony       0
category       0
film           0
winner         0
imdb_rating    1
Metascore      2
Tomatometer    4
dtype: int64

Data types of each column:
ceremony         int64
category        object
film            object
winner            bool
imdb_rating    float64
Metascore      float64
Tomatometer     object
dtype: object

Summary statistics for numerical columns:
         ceremony  imdb_rating   Metascore
count  186.000000   185.000000  184.000000
mean    85.650538     7.683784   81.385870
std      6.908000     0.480100   10.725488
min     72.000000     5.000000   41.000000
25%     81.000000     7.400000   76.000000
50%     86.000000     7.700000   83.000000
75%     91.750000     8.000000   89.000000
max     96.000000     9.000000  100.000000

Counts for categorical columns:
BEST PICTURE    186
Name: category, dtype: int64


In [101]:
# Update the information for the film "Precious: Based on the Novel 'Push' by Sapphire"
df.loc[df['film'] == "Precious: Based on the Novel 'Push' by Sapphire", 'film'] = "Precious"
df.loc[df['film'] == "Precious", 'imdb_rating'] = 7.3
df.loc[df['film'] == "Precious", 'Tomatometer'] = "97%"  # Store as a string with the '%' symbol
df.loc[df['film'] == "Precious", 'Metascore'] = 78



# Save the updated DataFrame to a new CSV file
df.to_csv("/workspace/oscar/cleaned_best_picture_2000.csv", index=False)

# Check the updated information
print(df[df['film'] == "Precious"])


    ceremony      category      film  winner  imdb_rating  Metascore  \
56        82  BEST PICTURE  Precious   False          7.3       78.0   

   Tomatometer  
56         97%  


In [102]:
# Update information for multiple films

# Moulin Rouge - Update Metascore
df.loc[df['film'] == "Moulin Rouge", 'Metascore'] = 66

# Don't Look Up - Update Tomatometer to 56%
df.loc[df['film'] == "Don't Look Up", 'Tomatometer'] = "56%"  # Store as a string with the '%' symbol

# Tár - Update Tomatometer to 91%
df.loc[df['film'] == "Tár", 'Tomatometer'] = "91%"  # Store as a string with the '%' symbol

# Maestro - Update Tomatometer to 78%
df.loc[df['film'] == "Maestro", 'Tomatometer'] = "78%"  # Store as a string with the '%' symbol

# Save the updated DataFrame to a new CSV file
df.to_csv("/workspace/oscar/updated_best_picture_2000.csv", index=False)

# Check the updated information
print(df[df['film'].isin(["Moulin Rouge", "Don't Look Up", "Tár", "Maestro"])])


     ceremony      category           film  winner  imdb_rating  Metascore  \
14         74  BEST PICTURE   Moulin Rouge   False          7.6       66.0   
158        94  BEST PICTURE  Don't Look Up   False          7.2       49.0   
172        95  BEST PICTURE            Tár   False          7.4       93.0   
181        96  BEST PICTURE        Maestro   False          6.5       77.0   

    Tomatometer  
14          73%  
158         56%  
172         91%  
181         78%  


In [106]:
import pandas as pd

# Load csv1 and csv2 with a different encoding to avoid UnicodeDecodeError
csv1 = pd.read_csv('/workspace/oscar/gpt1.csv', encoding='ISO-8859-1')  # Update with your csv1 file path
print(csv1.columns)


Index(['Gladiator'], dtype='object')


In [107]:
import pandas as pd

# Load csv1 and csv2 with a different encoding to avoid UnicodeDecodeError
csv1 = pd.read_csv('/workspace/oscar/gpt1.csv', encoding='ISO-8859-1')  # Update with your csv1 file path
csv2 = pd.read_csv('/workspace/oscar/updated_best_picture_2000.csv', encoding='ISO-8859-1')  # Update with your csv2 file path

# Create a new column in csv2 called 'nominated' with default value as empty
csv2['nominated'] = ''

# Loop through each film in csv1
for film in csv1['film_name_column']:  # Replace 'film_name_column' with the actual column name in csv1
    # Check if the film is in csv2 under the 'film' column
    if film in csv2['film'].values:
        # Update the new 'nominated' column to 'nominated' where the film matches
        csv2.loc[csv2['film'] == film, 'nominated'] = 'nominated'

# Save the updated csv2 to a new CSV file
csv2.to_csv('updated_csv2.csv', index=False)  # This will save the updated csv2 to a new file


In [109]:
import pandas as pd

# Load the updated csv2
csv2 = pd.read_csv('/workspace/oscar/updated_csv2.csv', encoding='ISO-8859-1')  # Make sure to use the correct encoding if needed

# Step 1: Replace empty values in 'oldvalue' column with 'none'
csv2['oldvalue'] = csv2['oldvalue'].fillna('none')

# Step 2: Compare 'oldvalue' and 'BAFTAs' columns
for index, row in csv2.iterrows():
    if row['BAFTAs'] != 'won':  # Only overwrite if 'BAFTAs' doesn't have the value 'won'
        csv2.at[index, 'BAFTAs'] = row['oldvalue']  # Overwrite 'BAFTAs' with 'oldvalue'

# Step 3: Save the updated csv2 to a new CSV file
csv2.to_csv('updated_with_changes.csv', index=False)  # This will save the modified csv2 to a new file


In [133]:
import pandas as pd

# Load data
df = pd.read_csv("/workspace/oscar/updated_with_changes.csv")

# Clean 'winner' column
df['winner'] = df['winner'].astype(int)

# Clean 'Tomatometer' (remove % and convert to float)
df['Tomatometer'] = df['Tomatometer'].str.replace('%', '').astype(float) / 100

# Encode GoldenGlobe and BAFTAs
award_mapping = {'won': 2, 'nominated': 1, 'none': 0}
df['GoldenGlobe'] = df['GoldenGlobe'].str.split().str[0].map(award_mapping).fillna(0)
df['BAFTAs'] = df['BAFTAs'].map(award_mapping).fillna(0)

In [134]:
# Create Total Awards Score
df['Total_Awards'] = df['GoldenGlobe'] + df['BAFTAs']

# Drop unnecessary columns
df = df.drop(['category', 'film'], axis=1)

In [135]:
# Assuming 'ceremony' column exists (if not, reset index)
train = df[df['ceremony'] <= 90]
test = df[df['ceremony'] > 90]

# Separate features (X) and target (y)
X_train = train.drop('winner', axis=1)
y_train = train['winner']
X_test = test.drop('winner', axis=1)
y_test = test['winner']

In [136]:
from sklearn.linear_model import LogisticRegression

# Initialize and train the model
model = LogisticRegression()
model.fit(X_train, y_train)

# Check accuracy on test data
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.89


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [137]:
from sklearn.metrics import classification_report

# Predict on test data
y_pred = model.predict(X_test)

# Generate evaluation report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94        49
           1       0.00      0.00      0.00         6

    accuracy                           0.89        55
   macro avg       0.45      0.50      0.47        55
weighted avg       0.79      0.89      0.84        55



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [138]:
# Example: New data for 2024 nominees
new_data = pd.DataFrame({
    'ceremony': [97, 97, 97],  # example default values for the new data
    'imdb_rating': [8.1, 7.9, 7.5],
    'Metascore': [85, 90, 78],
    'Tomatometer': [0.92, 0.88, 0.85],
    'GoldenGlobe': [2, 1, 0],
    'BAFTAs': [2, 1, 1],
    'Total_Awards': [4, 2, 1]
})


# Predict probabilities
probabilities = model.predict_proba(new_data)[:, 1]
print("Win probabilities:", probabilities)

Win probabilities: [0.36234901 0.01774957 0.00081818]


In [ ]:
# Example: New data for 2024 nominees
new_data = pd.DataFrame({
    'ceremony': [97, 97, 97, 97, 97, 97, 97, 97, 97, 97],
    'imdb_rating': [7.7, 7.8, 7.6, 7.4, 8.5, 5.5, 8.8, 7.2, 7.3, 7.6],
    'Metascore': [91, 90, 70, 79, 79, 70, 48, 91, 78, 73],
    'Tomatometer': [0.94, 0.94, 0.70, 0.79, 0.79, 0.70, 0.48, 0.91, 0.78, 0.73],
    'GoldenGlobe': [1, 2, 1, 1, 1, 2, 1, 1, 1, 1],
    'BAFTAs': [1, 1, 1, 2, 0, 1, 0, 0, 0, 0],
    'Total_Awards': [2, 3, 2, 3, 1, 3, 1, 1, 1, 1]
})



# Predict probabilities
probabilities = model.predict_proba(new_data)[:, 1]
print("Win probabilities:", probabilities)

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- ceremony


In [126]:
import pandas as pd

# Load the dataset
file_path = '/workspace/oscar/2025.csv'
df = pd.read_csv(file_path, encoding='ISO-8859-1')

# Display the DataFrame
display(df)


,ceremony,category,film,winner,imdb_rating,Metascore,Tomatometer,GoldenGlobe,BAFTAs
0,97,BEST PICTURE,Anora,NaN,NaN,NaN,NaN,NaN,NaN
1,97,BEST PICTURE,The Brutalist,NaN,NaN,NaN,NaN,NaN,NaN
2,97,BEST PICTURE,A Complete Unknown,NaN,NaN,NaN,NaN,NaN,NaN
3,97,BEST PICTURE,Conclave,NaN,NaN,NaN,NaN,NaN,NaN
4,97,BEST PICTURE,Dune: Part Two,NaN,NaN,NaN,NaN,NaN,NaN
5,97,BEST PICTURE,Emilia P¨¦rez,NaN,NaN,NaN,NaN,NaN,NaN
6,97,BEST PICTURE,I'm Still Here,NaN,NaN,NaN,NaN,NaN,NaN
7,97,BEST PICTURE,Nickel Boys,NaN,NaN,NaN,NaN,NaN,NaN
8,97,BEST PICTURE,The Substance,NaN,NaN,NaN,NaN,NaN,NaN
9,97,BEST PICTURE,Wicked,NaN,NaN,NaN,NaN,NaN,NaN


In [128]:
import pandas as pd
from imdb import Cinemagoer
import time
import requests

# Create Cinemagoer object
ia = Cinemagoer()

# Define OMDb API key (you can get a free API key at https://www.omdbapi.com/)
OMDB_API_KEY = '2121a3ae'

# Read CSV file
year2025_df = pd.read_csv('/workspace/oscar/2025.csv', encoding='ISO-8859-1')  # Replace with your file path

# Create new columns for IMDb ratings and box office
year2025_df['imdb_rating'] = None
year2025_df['box_office'] = None

# Set to track movies we've already processed
processed_movies = set()

# Function to fetch box office data from OMDb API
def get_box_office(movie_name):
    try:
        # Request movie data from OMDb API
        response = requests.get(f"http://www.omdbapi.com/?t={movie_name}&apikey={OMDB_API_KEY}")
        data = response.json()
        
        if data['Response'] == 'True':
            return data.get('BoxOffice', None)  # Return BoxOffice value if available
        else:
            print(f"Error fetching box office for {movie_name}: {data.get('Error', 'Unknown error')}")
            return None
    except Exception as e:
        print(f"Error fetching box office for {movie_name}: {e}")
        return None

# Iterate through each film in year2000_df DataFrame
for index, row in year2025_df.iterrows():
    movie_name = row['film']
    
    # Check if movie_name is valid (not NaN or empty)
    if not isinstance(movie_name, str) or not movie_name.strip():
        print(f"Skipping invalid movie name at index {index}: {movie_name}")
        continue  # Skip empty or invalid movie names
    
    # Skip duplicate movies (those already processed)
    if movie_name in processed_movies:
        print(f"Skipping already processed movie: {movie_name}")
        continue  # Skip already processed movies
    
    try:
        # Search for the movie
        movies = ia.search_movie(movie_name)
        
        if not movies:
            print(f"No results found for: {movie_name}")
            continue
            
        # Get first result and fetch movie details
        movie = ia.get_movie(movies[0].movieID)
        
        # Extract IMDb rating if available
        if 'rating' in movie.keys():
            year2025_df.at[index, 'imdb_rating'] = movie['rating']
            print(f"Found rating {movie['rating']} for {movie_name}")
        else:
            print(f"No rating available for: {movie_name}")
        
        # Fetch box office data from OMDb API
        box_office = get_box_office(movie_name)
        if box_office:
            year2025_df.at[index, 'box_office'] = box_office
            print(f"Found box office {box_office} for {movie_name}")
        else:
            print(f"No box office data available for: {movie_name}")
        
        # Mark this movie as processed
        processed_movies.add(movie_name)
            
    except Exception as e:
        print(f"Error processing {movie_name}: {str(e)}")
    
    # Add delay to prevent rate limiting
    time.sleep(1)  # Be nice to IMDb's servers

# Save updated DataFrame to a new CSV
year2025_df.to_csv('/workspace/oscar/updated_2025.csv', index=False)

# Display the first few rows of the updated DataFrame
year2025_df.head()  # You can also use display() in Jupyter if needed


Found rating 7.7 for Anora
Found box office $15,140,997 for Anora
Found rating 7.8 for The Brutalist
Found box office $12,116,310 for The Brutalist
Found rating 7.6 for A Complete Unknown
Found box office $66,837,359 for A Complete Unknown
Found rating 7.4 for Conclave
Found box office $31,795,645 for Conclave
Found rating 8.5 for Dune: Part Two
Found box office $282,144,358 for Dune: Part Two
Found rating 5.5 for Emilia P¨¦rez
Error fetching box office for Emilia P¨¦rez: Movie not found!
No box office data available for: Emilia P¨¦rez
Found rating 8.8 for I'm Still Here
Found box office $408,983 for I'm Still Here
Found rating 7.2 for Nickel Boys
Found box office $2,285,584 for Nickel Boys
Found rating 7.3 for The Substance
Found box office $16,774,179 for The Substance
Found rating 7.6 for Wicked
Found box office $465,905,925 for Wicked


,ceremony,category,film,winner,imdb_rating,Metascore,Tomatometer,GoldenGlobe,BAFTAs,box_office
0,97,BEST PICTURE,Anora,NaN,7.7,NaN,NaN,NaN,NaN,"$15,140,997"
1,97,BEST PICTURE,The Brutalist,NaN,7.8,NaN,NaN,NaN,NaN,"$12,116,310"
2,97,BEST PICTURE,A Complete Unknown,NaN,7.6,NaN,NaN,NaN,NaN,"$66,837,359"
3,97,BEST PICTURE,Conclave,NaN,7.4,NaN,NaN,NaN,NaN,"$31,795,645"
4,97,BEST PICTURE,Dune: Part Two,NaN,8.5,NaN,NaN,NaN,NaN,"$282,144,358"


In [130]:
import pandas as pd
import requests

API_KEY = "2121a3ae"  # Replace with your actual API key

def get_metascore_from_omdb(title):
    url = f"http://www.omdbapi.com/?t={title}&apikey={API_KEY}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return data.get("Metascore", "N/A")
    except Exception as e:
        print(f"Error fetching data for {title}: {e}")
    return "N/A"

# Load your CSV (ensure it has a 'film' column)
df = pd.read_csv("/workspace/oscar/updated_2025.csv", encoding='ISO-8859-1')
df["Metascore"] = df["film"].apply(get_metascore_from_omdb)
df.to_csv("movies_with_metascore.csv", index=False)

In [131]:
import pandas as pd
import requests

API_KEY = "2121a3ae"  # Replace with your actual API key

def get_rotten_tomatoes_rating(title):
    url = f"http://www.omdbapi.com/?t={title}&apikey={API_KEY}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            ratings = data.get("Ratings", [])
            for rating in ratings:
                if rating.get("Source") == "Rotten Tomatoes":
                    return rating.get("Value")
    except Exception as e:
        print(f"Error fetching data for {title}: {e}")
    return "N/A"

# Load your CSV (ensure it has a 'film' column)
df = pd.read_csv("/workspace/oscar/movies_with_metascore.csv")

# Apply the function to get Rotten Tomatoes rating
df["Tomatometer"] = df["film"].apply(get_rotten_tomatoes_rating)

# Save the updated dataframe to a new CSV file
df.to_csv("movies_with_metascore_and_tomatometer.csv", index=False)


In [132]:
import pandas as pd

# Load the dataset
file_path = '/workspace/oscar/movies_with_metascore_and_tomatometer.csv'
df = pd.read_csv(file_path, encoding='ISO-8859-1')

# Display the DataFrame
display(df)

,ceremony,category,film,winner,imdb_rating,Metascore,Tomatometer,GoldenGlobe,BAFTAs,box_office
0,97,BEST PICTURE,Anora,NaN,7.7,91,94%,nominated,nominated,"$15,140,997"
1,97,BEST PICTURE,The Brutalist,NaN,7.8,90,94%,won,nominated,"$12,116,310"
2,97,BEST PICTURE,A Complete Unknown,NaN,7.6,70,80%,nominated,nominated,"$66,837,359"
3,97,BEST PICTURE,Conclave,NaN,7.4,79,93%,nominated,won,"$31,795,645"
4,97,BEST PICTURE,Dune: Part Two,NaN,8.5,79,92%,nominated,none,"$282,144,358"
5,97,BEST PICTURE,Emilia P¨¦rez,NaN,5.5,70,72%,won,nominated,"$15,339,534"
6,97,BEST PICTURE,I'm Still Here,NaN,8.8,48,92%,nominated,none,"$408,983"
7,97,BEST PICTURE,Nickel Boys,NaN,7.2,91,90%,nominated,none,"$2,285,584"
8,97,BEST PICTURE,The Substance,NaN,7.3,78,89%,nominated,none,"$16,774,179"
9,97,BEST PICTURE,Wicked,NaN,7.6,73,88%,nominated,none,"$465,905,925"


In [ ]:
new_data = pd.DataFrame({
    'imdb_rating': [7.7, 7.8, 7.6, 7.4, 8.5, 5.5, 8.8, 7.2, 7.3, 7.6],
    'Metascore': [91, 90, 70, 79, 79, 70, 48, 91, 78, 73],
    'Tomatometer': [0.94, 0.94, 0.70, 0.79, 0.79, 0.70, 0.48, 0.91, 0.78, 0.73],
    'GoldenGlobe': [1, 2, 1, 1, 1, 2, 1, 1, 1, 1],
    'BAFTAs': [1, 1, 1, 2, 0, 1, 0, 0, 0, 0],
    'Total_Awards': [2, 3, 2, 3, 1, 3, 1, 1, 1, 1]
})
